In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from PIL import Image

In [ ]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

In [ ]:
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=10)
model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)

In [ ]:
for epoch in range(5):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, acc={train_acc:.4f}")

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

print(f"Test accuracy: {correct / total:.4f}")

In [ ]:
torch.save(model.state_dict(), "swin_mnist.pth")
print("✅ Model saved as swin_mnist.pth")


In [ ]:
model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=10)

model.load_state_dict(torch.load("swin_mnist.pth", map_location=device))

model.eval()

model.to(device)


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

def predict_image(image_path, device="cpu"):
    image = Image.open(image_path).convert("L")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)
    return pred.item(), conf.item()


In [ ]:
digit, confidence = predict_image("test/mnist_4.png", device)
print(f"Predicted digit: {digit}, confidence: {confidence:.2f}")

In [ ]:
digit, confidence = predict_image("test/mnist_0.png", device)
print(f"Predicted digit: {digit}, confidence: {confidence:.2f}")

In [ ]:
digit, confidence = predict_image("test/mnist_2.png", device)
print(f"Predicted digit: {digit}, confidence: {confidence:.2f}")

In [ ]:
digit, confidence = predict_image("test/mnist_7.png", device)
print(f"Predicted digit: {digit}, confidence: {confidence:.2f}")